# A2 — Pierce 1890 Knowledge-Base Demo
**Team G07 · doc-agent · Assignment 2**

This notebook provides graded A2 evidence:
1. **Part 1** — OCR quality metrics (CER, WER, Word-F1) on the 24 held-out pages vs. `grading_kit/labels.jsonl`
2. **Part 2** — Index overview (chunk count, dimension, pages indexed)
3. **Part 3** — Live vector retrieval demo with page citations and rendered figure images

> Run cells top to bottom after `bash scripts/build_index.sh` has completed.

In [1]:
import sys, json, re, html
from collections import Counter
from pathlib import Path

import yaml
import numpy as np

# Auto-detect repository root whether kernel started in . or ./notebooks
cwd = Path.cwd().resolve()
REPO = cwd.parent if not (cwd / 'configs' / 'config.yaml').is_file() and (cwd.parent / 'configs' / 'config.yaml').is_file() else cwd
sys.path.insert(0, str(REPO / 'src'))

cfg = yaml.safe_load((REPO / 'configs' / 'config.yaml').read_text())
INDEX_DIR     = (REPO / Path(cfg['index']['path'])).resolve()
cfg['index']['path'] = str(INDEX_DIR)  # Ensure absolute path for store.py

LABELS_PATH   = (REPO / 'grading_kit' / 'labels.jsonl').resolve()
CHANDRA_DIR   = (REPO / 'chandra').resolve()
PAGES_MD      = (REPO / 'chandra' / 'pages.md').resolve()
MINERU_PAGES  = (REPO / 'extras' / 'output' / 'mineru-ocr-full-book' / 'full-page' / 'pages.jsonl').resolve()
IMG_IDX_PATH  = INDEX_DIR / 'image_index.json'

print('REPO root :', REPO)
print('Labels    :', LABELS_PATH.is_file(), f'({LABELS_PATH})')
print('Index dir :', INDEX_DIR.is_dir(), f'({INDEX_DIR})')
print('MinerU    :', MINERU_PAGES.is_file())


REPO root : /Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter
Labels    : True (/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/grading_kit/labels.jsonl)
Index dir : True (/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/data/processed/index)
MinerU    : False


## Part 1 — OCR Quality on Held-Out Pages

In [2]:
if not LABELS_PATH.is_file():
    raise FileNotFoundError(f'Missing {LABELS_PATH} — add hand-corrected labels.')

labels = {}
for line in LABELS_PATH.read_text('utf-8').splitlines():
    if not line.strip() or line.lstrip().startswith('#'):
        continue
    row = json.loads(line)
    labels[row['page_id']] = row['text']

print(f'Loaded {len(labels)} held-out labels: {sorted(labels.keys())[:5]} ...')


Loaded 24 held-out labels: ['p0024', 'p0025', 'p0026', 'p0027', 'p0028'] ...


In [3]:
# Load OCR text per page (Canonical Corpus -> MinerU -> Chandra fallback)
from doc_agent.index.chunk import load_from_canonical_jsonl, load_from_mineru_jsonl, load_from_pages_markdown

canonical_candidates = [
    REPO / "data" / "canonical-pages.jsonl",
    REPO / "extras" / "indexing-benchmarks" / "data" / "canonical-pages.jsonl",
]
canonical_file = next((p for p in canonical_candidates if p.is_file()), None)

if canonical_file:
    page_chunks, _ = load_from_canonical_jsonl(canonical_file, cfg["ingest"]["doc_id"])
    ocr_engine_name = "Canonical Verified OCR (SOTA)"
elif MINERU_PAGES.is_file():
    page_chunks, _ = load_from_mineru_jsonl(MINERU_PAGES, cfg["ingest"]["doc_id"])
    ocr_engine_name = "MinerU Full-Page (SOTA)"
else:
    page_chunks, _ = load_from_pages_markdown(PAGES_MD, cfg["ingest"]["doc_id"])
    ocr_engine_name = "Chandra Vision"

ocr_pages = {c.page_ids[0]: c.text for c in page_chunks}
print(f"Total pages indexed: {len(ocr_pages)} ({ocr_engine_name})")

# ─── Metric helpers ───────────────────────────────────────────────────────────
def normalize(t): return re.sub(r"\s+", " ", t).strip().lower()

def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b)+1))
    for i, av in enumerate(a, 1):
        cur = [i]
        for j, bv in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[-1]+1, prev[j-1]+(av!=bv)))
        prev = cur
    return prev[-1]

def word_f1(hyp, ref):
    h, r = Counter(normalize(hyp).split()), Counter(normalize(ref).split())
    tp = sum((h & r).values())
    if not h and not r: return 1.0
    if not tp: return 0.0
    p, rc = tp/sum(h.values()), tp/sum(r.values())
    return 2*p*rc/(p+rc)

# ─── Score each held-out page ────────────────────────────────────────────────
results = []
for pid, ref in sorted(labels.items()):
    hyp  = ocr_pages.get(pid, "")
    ref_n, hyp_n = normalize(ref), normalize(hyp)
    cer = levenshtein(list(hyp_n), list(ref_n)) / max(len(ref_n), 1)
    wer = levenshtein(hyp_n.split(), ref_n.split()) / max(len(ref_n.split()), 1)
    f1  = word_f1(hyp, ref)
    results.append({"page_id": pid, "cer": cer, "wer": wer, "word_f1": f1})

macro_f1 = sum(r["word_f1"] for r in results) / max(len(results), 1)
micro_cer = sum(r["cer"] for r in results) / max(len(results), 1)
micro_wer = sum(r["wer"] for r in results) / max(len(results), 1)

print(f"\n{'Page':<10} {'CER':>7} {'WER':>7} {'Word F1':>9}")
print("-" * 38)
for r in results:
    print(f"{r['page_id']:<10} {r['cer']:>7.4f} {r['wer']:>7.4f} {r['word_f1']:>9.4f}")
print("-" * 38)
print(f"  MACRO AVG  {micro_cer:>7.4f} {micro_wer:>7.4f} {macro_f1:>9.4f}")
print(f"\nSample size: {len(results)} held-out pages ({ocr_engine_name})")


Total pages indexed: 1016 (Canonical Verified OCR (SOTA))



Page           CER     WER   Word F1
--------------------------------------
p0024       0.0126  0.0874    0.9305
p0025       0.0195  0.0732    0.9409
p0026       0.0020  0.0151    0.9864
p0027       0.1688  0.2143    0.9668
p0028       0.0023  0.0290    0.9784
p0029       0.2519  0.3368    0.9556
p0030       0.3559  0.4369    0.9323
p0031       0.2365  0.3343    0.9675
p0032       0.3784  0.5058    0.8186
p0033       0.1490  0.1761    0.9781
p0034       0.5041  0.5465    0.9596
p0035       0.0059  0.0691    0.9490
p0036       0.0011  0.0134    0.9900
p0037       0.0025  0.0259    0.9794
p0038       0.0058  0.0480    0.9633
p0039       0.1573  0.1511    0.9785
p0040       0.2373  0.3020    0.9697
p0041       0.1524  0.2778    0.8947
p0042       0.0004  0.0049    0.9963
p0043       0.1415  0.2222    0.9474
p0044       0.0000  0.0000    1.0000
p0045       0.1975  0.2731    0.9854
p0046       0.2170  0.2640    0.9592
p0047       0.0010  0.0091    0.9924
-----------------------------------

## Part 2 — Index Overview

In [4]:
from doc_agent.index.store import load

faiss_index, indexed_chunks, metadata = load(cfg)

# Load image index
img_idx = json.loads(IMG_IDX_PATH.read_text()) if IMG_IDX_PATH.exists() else {}
pages_with_figs = len(img_idx)
total_figs      = sum(len(v) for v in img_idx.values())

print('=' * 55)
print('  STAGE 4 — FAISS KNOWLEDGE BASE STATISTICS')
print('=' * 55)
print(f'  Index type         : {metadata["index_type"]}')
print(f'  Embedding model    : {cfg["embed"]["model"]}')
print(f'  Embedding dim      : {metadata["dimension"]}')
print(f'  Total chunks       : {metadata["count"]}')
print(f'  Chunk size (tokens): {cfg["index"]["chunk_tokens"]} (overlap {cfg["index"]["overlap"]})')
print(f'  Pages indexed      : {len(set(pid for c in indexed_chunks for pid in c.page_ids))}')
print(f'  Pages with figures : {pages_with_figs}  ({total_figs} total figure refs)')
print(f'  Index vectors      : {faiss_index.ntotal}')
print('=' * 55)


  STAGE 4 — FAISS KNOWLEDGE BASE STATISTICS
  Index type         : faiss:flat_ip
  Embedding model    : sentence-transformers/all-MiniLM-L6-v2
  Embedding dim      : 384
  Total chunks       : 1996
  Chunk size (tokens): 256 (overlap 32)
  Pages indexed      : 1016
  Pages with figures : 0  (0 total figure refs)
  Index vectors      : 1996


## Part 3 — Live Vector Retrieval Demo

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
from IPython.display import display, Image as IPImage, Markdown

embedder = SentenceTransformer(cfg['embed']['model'])

def retrieve(query: str, k: int = 5):
    """Embed query, search index, return top-k (chunk, score, images) tuples."""
    qvec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, positions = faiss_index.search(qvec, k)
    hits = []
    for score, pos in zip(scores[0], positions[0]):
        if pos < 0: continue
        chunk = indexed_chunks[pos]
        page_id = chunk.page_ids[0]
        page_imgs = img_idx.get(page_id, [])
        hits.append({'chunk': chunk, 'score': float(score), 'images': page_imgs})
    return hits

def show_results(query: str, k: int = 3):
    display(Markdown(f'### Query: *{query}*'))
    hits = retrieve(query, k)
    for i, h in enumerate(hits, 1):
        c = h['chunk']
        display(Markdown(
            f'**Result {i}** | Page: `{c.page_ids[0]}` | Cosine score: `{h["score"]:.4f}`\n\n'
            f'> {c.text[:400]}...'
        ))
        for img_meta in h['images'][:2]:  # show up to 2 figures per page
            img_path = CHANDRA_DIR / img_meta['webp']
            if img_path.exists():
                print(f'  📷 {img_meta["caption"][:100]}')
                display(IPImage(str(img_path), width=450))
        print()

# ─── Demo queries ──────────────────────────────────────────────────────────
show_results('What are the superficial muscles of the chest and abdomen?')


/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Query: *What are the superficial muscles of the chest and abdomen?*

**Result 1** | Page: `p0043` | Cosine score: `0.7053`

> THE MUSCLES. 35 Fig. 25. A representation of the superficial layer of muscles on the posterior portion of the body....

**Result 2** | Page: `p0041` | Cosine score: `0.6951`

> THE MUSCLES. 33 Fig. 24. ▲ representation of the superficial layer of muscles on the anterior portion of the body....

**Result 3** | Page: `p0038` | Cosine score: `0.5416`

> CHAPTER III. PHYSIOLOGICAL ANATOMY. THE MUSCLES. The Muscles are those organs of the body by which motion is produced, and are commonly known as flesh . A muscle is composed of fasciculi , or bundles of fibers, parallel to one another. They are soft, varying in size, of a reddish color, and inclosed in a cellular, membranous sheath. Each fasciculus contains a number of small fibers, which, when su...

In [6]:
show_results('What remedies are prescribed for inflammation and fever?')


### Query: *What remedies are prescribed for inflammation and fever?*

**Result 1** | Page: `p0409` | Cosine score: `0.6053`

> FEVER. 401 GENERAL PRINCIPLES FOR TREATMENT OF INFLAMMATION. Remove the exciting causes as far as practicable. If caused by a splinter or any foreign substance, it should be withdrawn, and if the injury is merely local, apply cold water to the parts to subdue the inflammation. If caused by a rabid animal, the wound should be enlarged and cupped, and the parts cleansed or destroyed by caustic. The ...

**Result 2** | Page: `p0406` | Cosine score: `0.5512`

> redder, more congested, and more painful than is natural. Inflammation is limited to certain parts, while fever influences the system generally. Inflammation gives rise to new formations, morbid products, and lesions, or alterations of structure. The morbid products of fever, and its modification of fluids are carried away by the secretions and excretions. The susceptibility of the body to inflamm...

**Result 3** | Page: `p0560` | Cosine score: `0.5420`

> than is natural. There is pain, attended by soreness upon pressure, and the patient becomes emaciated. Inflammation of the peritoneum is frequently an accompaniment of puerperal fever , which is a disease peculiar to childbirth, and which may arise from cold, or be communicated from one parturient patient to another by midwives. Treatment. In the remedial management of acute peritonitis, it is obv...

In [7]:
show_results('What is the structure of a nucleated cell?')


### Query: *What is the structure of a nucleated cell?*

**Result 1** | Page: `p0027` | Cosine score: `0.5439`

> CHAPTER II. PHYSIOLOGICAL ANATOMY. THE BONES. All living bodies are made up of tissues. There is no part, no organ, however soft and yielding, or hard and resisting, which has not this peculiarity of structure. The bones of animals, as well as their flesh and fat, are composed of tissues, and all alike made up of cells. When viewed under a microscope, each cell is seen to consist of three distinct...

**Result 2** | Page: `p0062` | Cosine score: `0.4289`

> fluid and a spheroidal vesicle, which is termed the nucleus . They have been regarded by some physiologists as identical with those of the lymph and chyle. Dr. Carpen- ter believes that the function of these cells is to convert albumen into fibrin, by the simple process of cell-growth. It is generally believed that the red corpuscles are derived in some way from the colorless. It is supposed that ...

**Result 3** | Page: `p0475` | Cosine score: `0.3980`

> as lip, breast, womb, skin, and bone cancers. These different varieties may exist separately, or may be combined, so that several varieties may appear in a single growth; or, according to J. Hughes Bennett, Frank H. Hamilton, and others, they are sometimes transformed one into another. In consequence, it is sometimes very difficult to distinguish and classify them. The cancer-cells present almost ...

In [8]:
# ─── Abstention & Negative Evaluation ───────────────────────────────────────
ABSTENTION_THRESHOLD = 0.40  # Calibrated in Stage 4 offline dev grid

def evaluate_abstention(query: str, threshold: float = ABSTENTION_THRESHOLD):
    display(Markdown(f'### Out-of-Corpus / Negative Query: *{query}*'))
    hits = retrieve(query, k=1)
    if not hits or hits[0]['score'] < threshold:
        top_score = hits[0]['score'] if hits else 0.0
        top_page = hits[0]['chunk'].page_ids[0] if hits else 'None'
        display(Markdown(
            f'🛑 **ABSTAINED (Decision: OUT_OF_CORPUS)**\n\n'
            f'- Nearest confidence score `{top_score:.4f}` is below threshold $\\tau = {threshold:.2f}$.\n'
            f'- Nearest (weak) match was on `{top_page}`, but correctly rejected to avoid hallucination.'
        ))
    else:
        display(Markdown(f'⚠️ Matched with score `{hits[0]["score"]:.4f}` on `{hits[0]["chunk"].page_ids[0]}`.'))

evaluate_abstention('What are the clinical symptoms and mRNA vaccine protocols for COVID-19?')
evaluate_abstention('How do silicon semiconductor transistors function in CPUs?')


### Out-of-Corpus / Negative Query: *What are the clinical symptoms and mRNA vaccine protocols for COVID-19?*

⚠️ Matched with score `0.4733` on `p0530`.

### Out-of-Corpus / Negative Query: *How do silicon semiconductor transistors function in CPUs?*

🛑 **ABSTAINED (Decision: OUT_OF_CORPUS)**

- Nearest confidence score `0.1650` is below threshold $\tau = 0.40$.
- Nearest (weak) match was on `p0149`, but correctly rejected to avoid hallucination.

## A2 Form — Numbers to Copy
Copy the values from the cells above directly into **Section 5** of `forms/A2_form.docx`.

| Metric | Value |
|---|---|
| OCR sample size | *from Part 1 output* |
| Macro Word-F1 | *from Part 1 output* |
| Micro CER | *from Part 1 output* |
| Chunks indexed | *from Part 2 output* |
| Embedding dim | 384 (all-MiniLM-L6-v2) |
| Index type | faiss:flat_ip |
| Pages with figures | *from Part 2 output* |
| Query shown | *from Part 3 output* |
| Top-1 page citation | *from Part 3 output* |